In [ ]:
# Cell 1 - Environment Setup & Logging Init (TF Only)
import os
import sys
import gc
import time
import json
import random
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.applications import MobileNet, MobileNetV2, MobileNetV3Small, MobileNetV3Large
from tensorflow.keras import layers, models, callbacks

# ── scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

RESULTS_FILE = "mobilenet_results.json"
CHECKPOINT_DIR = "checkpoints"
LOG_FILE = "training.log"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Setup Logging
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="a"
)
logger = logging.getLogger()
logger.info("--- PHIÊN LÀM VIỆC TF MỚI BẮT ĐẦU ---")

def clear_runtime_memory(verbose=True):
    """Giải phóng RAM và dọn dẹp Graph trệt để cho TensorFlow."""
    tf.keras.backend.clear_session()
    gc.collect()
    if verbose:
        print(">>> TensorFlow Session & RAM đã được dọn dẹp sạch sẽ.")

def save_result_to_disk(model_name, metrics, history):
    data = {}
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, "r") as f:
            try: data = json.load(f)
            except: data = {}
    data[model_name] = {
        "accuracy": float(metrics["accuracy"]),
        "f1": float(metrics["f1"]),
        "train_time": float(metrics["train_time"]),
        "acc_history": [float(x) for x in history["accuracy"]],
        "val_acc_history": [float(x) for x in history["val_accuracy"]]
    }
    with open(RESULTS_FILE, "w") as f:
        json.dump(data, f, indent=4)

print(f"TensorFlow {tf.__version__} initialized.")

In [ ]:
# Cell 2 - Load & Prepare Data (uint8 for RAM saving)
clear_runtime_memory()
(x_train_orig, y_train_orig), (x_test_orig, y_test_orig) = tf.keras.datasets.cifar10.load_data()
x_all = np.concatenate([x_train_orig, x_test_orig], axis=0)
y_all = np.concatenate([y_train_orig, y_test_orig], axis=0).reshape(-1)
del x_train_orig, x_test_orig, y_train_orig, y_test_orig

print("Resizing to 96x96 (uint8)...")
X_96 = np.empty((60000, 96, 96, 3), dtype=np.uint8)
for i in range(0, 60000, 5000):
    batch = x_all[i:i+5000].astype("float32")
    X_96[i:i+5000] = tf.cast(tf.image.resize(batch, (96, 96)), tf.uint8).numpy()
del x_all
gc.collect()

X_train, X_temp, y_train, y_temp = train_test_split(X_96, y_all, test_size=0.3, random_state=SEED, stratify=y_all)
del X_96
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)
del X_temp
gc.collect()

DEMO_EPOCHS = 36
BATCH_SIZE = 64
print("Dữ liệu đã sẵn sàng.")

In [ ]:
# Cell 3 - TF MobileNetV1
clear_runtime_memory()
checkpoint_path = os.path.join(CHECKPOINT_DIR, "MobileNetV1_best.h5")
last_path = os.path.join(CHECKPOINT_DIR, "MobileNetV1_last.h5")

base = MobileNet(weights="imagenet", include_top=False, input_shape=(96, 96, 3))
base.trainable = False
model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./255),
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Resume logic
if os.path.exists(last_path):
    print(f"Đang tải lại checkpoint từ {last_path}")
    model.load_weights(last_path)

cb_best = callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
cb_last = callbacks.ModelCheckpoint(last_path, save_best_only=False)

start_t = time.time()
h = model.fit(X_train, y_train, validation_data=(X_val, y_val), 
              epochs=DEMO_EPOCHS, batch_size=BATCH_SIZE, verbose=1,
              callbacks=[cb_best, cb_last])
t_time = time.time() - start_t

# Evaluation
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
_, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

save_result_to_disk("MobileNetV1", {"accuracy": acc, "f1": f1, "train_time": t_time}, h.history)
del model, base, h
clear_runtime_memory()

In [ ]:
# Cell 4 - TF MobileNetV2
clear_runtime_memory()
checkpoint_path = os.path.join(CHECKPOINT_DIR, "MobileNetV2_best.h5")
last_path = os.path.join(CHECKPOINT_DIR, "MobileNetV2_last.h5")

base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(96, 96, 3))
base.trainable = False
model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./255),
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Resume logic
if os.path.exists(last_path):
    print(f"Đang tải lại checkpoint từ {last_path}")
    model.load_weights(last_path)

cb_best = callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
cb_last = callbacks.ModelCheckpoint(last_path, save_best_only=False)

start_t = time.time()
h = model.fit(X_train, y_train, validation_data=(X_val, y_val), 
              epochs=DEMO_EPOCHS, batch_size=BATCH_SIZE, verbose=1,
              callbacks=[cb_best, cb_last])
t_time = time.time() - start_t

# Evaluation
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
_, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

save_result_to_disk("MobileNetV2", {"accuracy": acc, "f1": f1, "train_time": t_time}, h.history)
del model, base, h
clear_runtime_memory()

In [ ]:
# Cell 5 - TF MobileNetV3_Small
clear_runtime_memory()
checkpoint_path = os.path.join(CHECKPOINT_DIR, "MobileNetV3_Small_best.h5")
last_path = os.path.join(CHECKPOINT_DIR, "MobileNetV3_Small_last.h5")

base = MobileNetV3Small(weights="imagenet", include_top=False, input_shape=(96, 96, 3))
base.trainable = False
model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./255),
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Resume logic
if os.path.exists(last_path):
    print(f"Đang tải lại checkpoint từ {last_path}")
    model.load_weights(last_path)

cb_best = callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
cb_last = callbacks.ModelCheckpoint(last_path, save_best_only=False)

start_t = time.time()
h = model.fit(X_train, y_train, validation_data=(X_val, y_val), 
              epochs=DEMO_EPOCHS, batch_size=BATCH_SIZE, verbose=1,
              callbacks=[cb_best, cb_last])
t_time = time.time() - start_t

# Evaluation
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
_, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

save_result_to_disk("MobileNetV3_Small", {"accuracy": acc, "f1": f1, "train_time": t_time}, h.history)
del model, base, h
clear_runtime_memory()

In [ ]:
# Cell 6 - TF MobileNetV3_Large
clear_runtime_memory()
checkpoint_path = os.path.join(CHECKPOINT_DIR, "MobileNetV3_Large_best.h5")
last_path = os.path.join(CHECKPOINT_DIR, "MobileNetV3_Large_last.h5")

base = MobileNetV3Large(weights="imagenet", include_top=False, input_shape=(96, 96, 3))
base.trainable = False
model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    layers.Rescaling(1./255),
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Resume logic
if os.path.exists(last_path):
    print(f"Đang tải lại checkpoint từ {last_path}")
    model.load_weights(last_path)

cb_best = callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
cb_last = callbacks.ModelCheckpoint(last_path, save_best_only=False)

start_t = time.time()
h = model.fit(X_train, y_train, validation_data=(X_val, y_val), 
              epochs=DEMO_EPOCHS, batch_size=BATCH_SIZE, verbose=1,
              callbacks=[cb_best, cb_last])
t_time = time.time() - start_t

# Evaluation
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
_, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

save_result_to_disk("MobileNetV3_Large", {"accuracy": acc, "f1": f1, "train_time": t_time}, h.history)
del model, base, h
clear_runtime_memory()

In [ ]:
# Cell 7 - Final Summary
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        data = json.load(f)
    df = pd.DataFrame([{"Model": k, "Acc": v["accuracy"], "F1": v["f1"], "Time(s)": v["train_time"]} for k,v in data.items()])
    display(df.sort_values("Acc", ascending=False))
    
    plt.figure(figsize=(10, 5))
    for name, v in data.items():
        plt.plot(v["val_acc_history"], label=name)
    plt.title("TF Models Validation Accuracy"); plt.legend(); plt.show()
else:
    print("Kết quả không tìm thấy.")